In [1]:
# Auto reload
%load_ext autoreload
%autoreload 2

# library imports
from matplotlib import pyplot as plt
import numpy as np

# import PGL Eyelink interface
from pgl import pglEyelink, pglEyelinkData, pglTask, pglParameter

# Load PGL libraries and start a PGL window
from pgl import pgl, pglExperiment
pgl = pgl()
e = pglExperiment(pgl, settingsName="ViewPixx")

# close any existing windows
pgl.cleanUp()


================================ pglBase: init =================================
(pgl) mglMetal error log can be viewed in MacOS Console app by searching for PROCESS mglMetal or in a terminal with:
      log stream --level info --process mglMetal
(pgl) To search for something specifc, e.g. messages from mglMovie:
      log stream --predicate 'eventMessage CONTAINS "mglMovie"' --style syslog --level info
(pgl:checkOS) Python version: 3.12.3 | packaged by conda-forge | (main, Apr 15 2024, 18:35:20) [Clang 16.0.6 ]
(pgl:checkOS) Running on MacBook Pro (MacBookPro18,3) with macOS version: 26.4.1
(pgl:checkOS) Apple M1 Pro Cores: 8 (6 Performance and 2 Efficiency) Memory: 32 GB
(pgl:checkOS) GPU: Apple M1 Pro (Built-In) 14 cores, Metal 4 support
(pgl:checkOS)   Color LCD [Main Display]: 3024 x 1964 Retina (Built-in Liquid Retina XDR Display) GammaTable size: 1024
(pglBase) Main library instance created
(pglSettingsManager:loadSettings) Settings file '/Users/justin/.pgl/settings/ViewPixx.jso

In [ ]:

e.initScreen()
#eyelink = pglEyelink(pgl)

In [ ]:
# TODO
# 1 abort calibration key
# *** 2 Eat all keys
# *** 3 setting for eccentricity of calibration
# 4 Check Read / Write EDF
# 5 Send triggerq to EDF 
# 6 Handle more gracefully stopping keyboard - with interrupt handler for eyelink
# 7 Force a
# qQccept of point
# 8 Display crosses on eye screen
# 9 
eyelink.setCustomCalibrationPoints(margin=0.7, numPoints=9)
eyelink.calibrate()
e.endScreen()
eyelink.openEDF('test')
eyelink.start()
pgl.waitSecs(2)
eyelink.sendMessage("Test 1")
pgl.waitSecs(2)
eyelink.sendMessage("Test 2")
pgl.waitSecs(1)
eyelink.stop()
x = eyelink.getEDF('test','/Users/justin/data')


In [ ]:
d=pglEyelinkData('/Users/justin/Desktop/test.asc')

In [ ]:
print(d)
plt.plot(d.samples['time'],d.samples['x'],'.')
#d.samples
#d.messages
#d.fixations
#d.saccades
d.blinks

In [ ]:
# Set up eye fixation task
class pgFixationTask(pglTask):
    
    ########################
    def __init__(self, pgl):
        super().__init__(pgl)
        
        # set task parameters, these will automatically be saved in the settings file
        self.settings.taskName = "Eye Fixation Task"
        self.settings.seglen = [1.5]
        
        # fixed parameters, these will automatically be saved in the settings file
        self.settings.fixedParameters = {
            'nPoints':9,
            'width':10,
            'height':10
        }        
        p = self.settings.fixedParameters

        # compute positions of calibration points
        if p['nPoints']==9:
            x = p['width']*0.5
            y = p['height']*0.5
            self.settings.fixedParameters['calibrationPoints'] = [(0,0),(-x,y),(0,y),(x,y),(-x,0),(x,0),(-x,-y),(0,-y),(x,-y)]
        else:
            raise ValueError(f"Unsupported number of points: {p['nPoints']}")
        
        fixIndex = pglParameter('fixIndex',np.arange(p['nPoints']))
        self.addParameter(fixIndex)
        
    ########################
    def updateScreen(self):
        if self.state.currentSegment==0:
            # get the current fix Index
            fixIndex = self.currentParams['fixIndex']
            # get the position
            x = self.settings.fixedParameters['calibrationPoints'][fixIndex][0]
            y = self.settings.fixedParameters['calibrationPoints'][fixIndex][1]
            self.pgl.circle(0.25,x,y,fill=True)
            self.pgl.fixationCross(0.25,x,y,color=[1,0,0])

# initialize task
fixTask = pgFixationTask(pgl)


In [ ]:
e.initScreen()
e.addTask(fixTask)
e.run()

================================= pglBase:open =================================
(pglBase:open) Starting mglMetal application: /Users/justin/Library/Developer/Xcode/DerivedData/Build/Products/Release/mglMetal.app
(pglBase:open) Using socket with address: /Users/justin/Library/Containers/gru.mglMetal/Data/pglMetal.socket.20260512_225846.HFHAxNgBYX
(pgl:_pglComm) .Connected to: /Users/justin/Library/Containers/gru.mglMetal/Data/pglMetal.socket.20260512_225846.HFHAxNgBYX
(pgl:_resolution:getResolution) Display 0/1: 1512x982 120Hz 32bits
(pglKeyboardMouse:start) Starting keyboard and mouse event listener.
(pglEventListener) Eating 7 keys: ['1', '2', '3', '4', '`', 'escape', 'space']
Block 1: 9 randomized over: ['fixIndex']
(Eye Fixation Task) Trial 1: fixIndex=4 
(pglExperiment:startPhase) Starting phase: 0/2
(pglExperiment:run) Experiment started.
(Eye Fixation Task) Trial 34: fixIndex=1 
(Eye Fixation Task) Trial 35: fixIndex=8 
(Eye Fixation Task) Trial 2: fixIndex=6 
(Eye Fixation Task

In [ ]:
eyelink.openEDF('test')
eyelink.start()
pgl.waitSecs(2)
eyelink.sendMessage("Test 1")
pgl.waitSecs(2)
eyelink.sendMessage("Test 2")
pgl.waitSecs(1)
eyelink.stop()
x = eyelink.getEDF('test','/Users/justin/data')

In [ ]:
help(pgl.circle)

In [ ]:
#from pgl import pglEyelink
pgl.cleanUp()
pgl.open(0,800,600)
pgl.visualAngle(57,30,20)
from pgl import pglEyelinkCustomDisplay
#eyelink = pglEyelink(pgl)
#customDisplay = pglEyelinkCustomDisplay(pgl,eyelink)

customDisplay = pglEyelinkCustomDisplay(pgl)

customDisplay.clear_cal_display()
customDisplay.draw_line(00,0,500,500,0)
customDisplay.draw_lozenge(100,200,100,300,0)
customDisplay.draw_cal_target(300,300)
#customDisplay.get_input_key()
#pgl.flush()

In [ ]:
eyelink.close()

In [ ]:
import pylink

def setup_eye_link(eyelink_address="100.1.1.1"):
    try:
        # Create an EyeLink tracker object.
        el_tracker = pylink.EyeLink(eyelink_address)
        print(f"Connecting to EyeLink at {eyelink_address}...")
        el_tracker.open()  # Open the connection to the EyeLink
        print("EyeLink connected.")

        # Check connection status
        if el_tracker.isConnected():
            print("EyeLink is connected.")
        else:
            print("EyeLink is not connected.")
            return

        # Setup and calibrate the tracker
        print("Performing tracker setup...")
        el_tracker.doTrackerSetup()  # Setup the tracker (calibration and validation)
        print("Tracker setup completed.")

        # Optionally, you could start recording data here
        # el_tracker.startRecording(1, 1, 1, 1)  # Start recording

        # Close the connection when finished
        el_tracker.close()
        print("EyeLink connection closed.")

    except pylink.EyeLinkError as e:
        print(f"An EyeLink error occurred: {e}")
    except Exception as e:
        print(f"An unexpected error occurred: {e}")

# Run the setup
setup_eye_link()